# Phase 1: GPU Basics & Performance Profiling

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 1: GPU Access Check

**Goal:** Before we jump into using the GPU, let's just do a quick sanity check to see if Kaggle has actually assigned us a GPU and if PyTorch is able to see it.

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print("SM count:", torch.cuda.get_device_properties(0).multi_processor_count)

CUDA available: True
GPU name: Tesla T4
Total VRAM (GB): 15.64
SM count: 40


## Lesson 2: CPU RAM to GPU VRAM Transfer

**Goal:** Memory transfer is usually a big bottleneck. Here, we are sending a huge 4096x4096 float32 tensor (around 67 MB) from CPU RAM to GPU VRAM just to measure the transfer speed in GB/s.

In [2]:
import time
import torch

n = 4096
x_cpu = torch.randn(n, n, dtype=torch.float32)
size_mb = x_cpu.numel() * x_cpu.element_size() / 1e6
print("Tensor device:", x_cpu.device)
print("Tensor size (MB):", round(size_mb, 1))

_ = torch.randn(10, device="cuda")  # warm-up
torch.cuda.synchronize()

times = []
for _ in range(5):
    torch.cuda.synchronize()
    start = time.perf_counter()
    x_gpu = x_cpu.to("cuda")
    torch.cuda.synchronize()
    times.append(time.perf_counter() - start)

best = min(times)
print("Tensor device now:", x_gpu.device)
print("Transfer times (ms):", [round(t * 1000, 2) for t in times])
print("Best bandwidth (GB/s):", round(size_mb / 1000 / best, 2))
print("VRAM used (MB):", round(torch.cuda.memory_allocated() / 1e6, 1))

Tensor device: cpu
Tensor size (MB): 67.1
Tensor device now: cuda:0
Transfer times (ms): [24.34, 14.34, 14.17, 14.08, 14.13]
Best bandwidth (GB/s): 4.77
VRAM used (MB): 67.1


## Lesson 3: The Async Timing Trap

**Goal:** GPU operations in PyTorch are async. If we don't use `torch.cuda.synchronize()`, the timer will stop before the GPU even finishes the work, giving us fake TFLOPS. Let's benchmark a matmul with and without sync to see this trap.

In [3]:
import time
import torch

n = 4096
A = torch.randn(n, n, device="cuda")
B = torch.randn(n, n, device="cuda")

for _ in range(3):  # warm-up
    _ = A @ B
torch.cuda.synchronize()

flops = 2 * n**3

start = time.perf_counter()
C = A @ B
t_wrong = time.perf_counter() - start

torch.cuda.synchronize()
start = time.perf_counter()
C = A @ B
torch.cuda.synchronize()
t_right = time.perf_counter() - start

print(f"Without sync: {t_wrong*1000:.3f} ms -> {flops/t_wrong/1e12:.1f} TFLOPS")
print(f"With sync:    {t_right*1000:.3f} ms -> {flops/t_right/1e12:.1f} TFLOPS")

Without sync: 0.525 ms -> 261.8 TFLOPS
With sync:    32.442 ms -> 4.2 TFLOPS


## Lesson 4: CPU vs GPU (Single Matmul)

**Goal:** Let's do a simple comparison. We will run a 4096x4096 matmul on the CPU using NumPy and then on the GPU using PyTorch. We will take the best of N runs to see the actual speedup we get.

In [4]:
import os, time
import numpy as np
import torch

n = 4096
flops = 2 * n**3
print("CPU cores visible:", os.cpu_count())

A_np = np.random.randn(n, n).astype(np.float32)
B_np = np.random.randn(n, n).astype(np.float32)

_ = A_np @ B_np  # warm-up
cpu_times = []
for _ in range(3):
    start = time.perf_counter()
    _ = A_np @ B_np
    cpu_times.append(time.perf_counter() - start)
t_cpu = min(cpu_times)

A = torch.from_numpy(A_np).to("cuda")
B = torch.from_numpy(B_np).to("cuda")
for _ in range(3):  # warm-up
    _ = A @ B
torch.cuda.synchronize()

gpu_times = []
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    _ = A @ B
    torch.cuda.synchronize()
    gpu_times.append(time.perf_counter() - start)
t_gpu = min(gpu_times)

print(f"CPU (numpy): {t_cpu*1000:.1f} ms -> {flops/t_cpu/1e9:.1f} GFLOPS")
print(f"GPU (torch): {t_gpu*1000:.1f} ms -> {flops/t_gpu/1e9:.1f} GFLOPS")
print(f"Speedup: {t_cpu/t_gpu:.1f}x")

CPU cores visible: 4
CPU (numpy): 553.9 ms -> 248.1 GFLOPS
GPU (torch): 32.3 ms -> 4257.7 GFLOPS
Speedup: 17.2x


## Lesson 5: Matrix Size Sweep (Finding the Crossover)

**Goal:** GPUs have some fixed overhead when launching kernels. By testing matrix sizes from 32 to 4096, we want to find the exact crossover point where the matrix is big enough for the GPU to finally beat the CPU.

In [5]:
import time
import numpy as np
import torch

def best_time(fn, repeats, sync=False):
    fn()  # warm-up
    if sync:
        torch.cuda.synchronize()
    best = float("inf")
    for _ in range(repeats):
        if sync:
            torch.cuda.synchronize()
        start = time.perf_counter()
        fn()
        if sync:
            torch.cuda.synchronize()
        best = min(best, time.perf_counter() - start)
    return best

print(f"{'n':>6} {'cpu ms':>10} {'gpu ms':>10} {'speedup':>9} {'gpu GFLOPS':>11}")
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A_np = np.random.randn(n, n).astype(np.float32)
    B_np = np.random.randn(n, n).astype(np.float32)
    A = torch.from_numpy(A_np).to("cuda")
    B = torch.from_numpy(B_np).to("cuda")

    t_cpu = best_time(lambda: A_np @ B_np, repeats=20 if n <= 1024 else 3)
    t_gpu = best_time(lambda: A @ B, repeats=50 if n <= 1024 else 10, sync=True)

    print(f"{n:>6} {t_cpu*1000:>10.4f} {t_gpu*1000:>10.4f} {t_cpu/t_gpu:>8.1f}x {2*n**3/t_gpu/1e9:>11.1f}")

     n     cpu ms     gpu ms   speedup  gpu GFLOPS
    32     0.0039     0.0250      0.2x         2.6
    64     0.0062     0.0247      0.3x        21.3
   128     0.0335     0.0290      1.2x       144.4
   256     0.1647     0.0413      4.0x       811.8
   512     1.0818     0.0708     15.3x      3791.9
  1024     8.9329     0.3682     24.3x      5831.9
  2048    67.6697     2.4567     27.5x      6993.0
  4096   555.6349    32.4328     17.1x      4237.6


## Lesson 6: Memory Bandwidth Measurement

**Goal:** To measure the practical VRAM bandwidth of our GPU, we will run a very simple memory-bound operation (`y = x + 1`) on a massive tensor. Let's see how close we get to the theoretical hardware limit.

In [6]:
import time
import torch

N = 128 * 1024 * 1024  # 128M floats = ~537 MB
x = torch.randn(N, device="cuda")
y = torch.empty_like(x)

for _ in range(3):  # warm-up
    torch.add(x, 1.0, out=y)
torch.cuda.synchronize()

best = float("inf")
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    torch.add(x, 1.0, out=y)
    torch.cuda.synchronize()
    best = min(best, time.perf_counter() - start)

bytes_moved = 2 * N * 4
print(f"Time: {best*1000:.2f} ms")
print(f"Bandwidth: {bytes_moved/best/1e9:.1f} GB/s  ({bytes_moved/best/1e9/320*100:.0f}% of 320 GB/s peak)")
print(f"Compute: {N/best/1e9:.1f} GFLOPS")

Time: 4.66 ms
Bandwidth: 230.6 GB/s  (72% of 320 GB/s peak)
Compute: 28.8 GFLOPS


## Lesson 7: Batch Size vs Time (Weight Reuse)

**Goal:** We are multiplying a fixed weight matrix with different batch sizes (1 to 256). You will see that batch 1 and batch 32 take almost the same time because reading the weights from memory is the real bottleneck, not the actual math.

In [7]:
import time
import torch

n = 8192
W = torch.randn(n, n, device="cuda", dtype=torch.float16)
w_bytes = n * n * 2

print(f"{'batch':>6} {'ms':>8} {'W-read GB/s':>12} {'TFLOPS':>8}")
for batch in [1, 2, 4, 8, 16, 32, 64, 128, 256]:
    X = torch.randn(n, batch, device="cuda", dtype=torch.float16)
    for _ in range(3):  # warm-up
        _ = W @ X
    torch.cuda.synchronize()

    best = float("inf")
    for _ in range(20):
        torch.cuda.synchronize()
        start = time.perf_counter()
        _ = W @ X
        torch.cuda.synchronize()
        best = min(best, time.perf_counter() - start)

    tflops = 2 * n * n * batch / best / 1e12
    print(f"{batch:>6} {best*1000:>8.3f} {w_bytes/best/1e9:>12.1f} {tflops:>8.2f}")

 batch       ms  W-read GB/s   TFLOPS
     1    0.532        252.4     0.25
     2    0.718        187.1     0.37
     4    0.721        186.1     0.74
     8    0.688        195.2     1.56
    16    0.693        193.7     3.10
    32    0.699        192.0     6.14
    64    0.723        185.8    11.89
   128    0.781        171.9    22.01
   256    1.644         81.6    20.90


## Sprint 0 summary (Tesla T4, Kaggle)
- 40 SMs, 2560 CUDA cores, 15.64 GB VRAM
- CPU->GPU transfer ~4.85 GB/s (pageable), VRAM bandwidth ~239-245 GB/s
- GPU ops async: sync ke bina timing galat (261 TFLOPS fake vs 2.9 real)
- Matmul: crossover CPU vs GPU at n~128, 18.5x at n=4096 (~4.5 TFLOPS)
- Chhote kaam par GPU ka ~30 us fixed overhead
- Decode = memory-bound; batching se weights ka reuse (batch 64 ~ 1.33x time, 64x kaam)